# Statistical analysis of the data

This notebook consists of statistical checks of the inspected variables, related to the hypothesis set for this part of analysis: 

RQ: How do selected manifestations of political identity (political parties, party status, political orientation) shape parliamentary discourse in terms of sentiment?

- H1. Coalition parties will consistently exhibit higher neutral/positive sentiment, while opposition parties will display more negativity, though intensity may vary across terms based on political events.
- H2: Changes in political orientation will correlate with sentiment shifts, with right/traditionalist parties showing higher negativity.
- H3: Negative sentiment will dominate parliamentary debates, with minor occurrences of other sentiment categories. Political crises, elections, or leadership changes will correlate with sentiment shifts.

The sample illustrates the workflow. The full analysis is run on the complete dataset for final results.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats


In [2]:
df = pd.read_csv("../Results/Datasets/ParlaMint-SI_Family.tsv", sep="\t", encoding="utf-8")
df.head()
df.shape

(3534, 26)

## H1: Coalition parties more neutral/positive; opposition more negative

Given the non-normal distributions of sentiment checks, we will use Mann–Whitney U test to check several options: 

- a) is there a (statistically significant) difference between the two variables?
    - Null hypothesis (H₀): The distribution of sentiment scores is the same in coalition and opposition parties.

For this we will first run two-sided test (to compare if there is a difference between coalition and opposition in terms of sentiment).

- b) if the results are statistically significant, we will check the direction with the one-sided test
- c) Since procedural speeches are, as seen in distribution check, mostly neutral in sentiment (Neutral - 94.56%, Negative - 4.53%, Positive - 0.89%),
 but the actual chairs are usually directly affiliated with political parties/parliamentary groups, we will run the test: 
    - first on raw sentiment scores, separated by coalition and opposition (keeping the dataset intact, inspecting full picture)
    - and for a second run, we will remove the procedural speeches from the dataset as a robustnes check and to see if results from first run still hold

### Two-sided Mann–Whitney U (entire dataset)

In [3]:
from scipy.stats import mannwhitneyu

coal_scores = df[df["Party_status"] == "Coalition"]["Senti_n"]
oppo_scores = df[df["Party_status"] == "Opposition"]["Senti_n"]

u_stat, p_two = mannwhitneyu(coal_scores, oppo_scores, alternative="two-sided")
print("Mann–Whitney U statistic:", u_stat)
print("Two-sided p-value:", p_two)

n1, n2 = len(coal_scores), len(oppo_scores)
effect_size_r = 1 - (2*u_stat) / (n1*n2)
print("Effect size (r):", effect_size_r)
print("Coalition median:", coal_scores.median())
print("Opposition median", oppo_scores.median())


Mann–Whitney U statistic: 1645543.0
Two-sided p-value: 3.046903445819607e-70
Effect size (r): -0.36230653041473726
Coalition median: 3.0
Opposition median 1.18


Sentiment scores differed significantly between coalition and opposition parties (Mann–Whitney U = 1,645,543, p < 0.001). The median sentiment was higher for coalition parties, consistent with the hypothesis that coalition MPs speak in a more neutral/positive tone compared to opposition MPs. 

Given the statistically significant results in the two-sided test, we then test for direction with a one-sided test.

### One-sided Mann-Whitney U test (entire dataset)


In [4]:
from itertools import product
u_stat, p_one = mannwhitneyu(coal_scores, oppo_scores, alternative='greater')

print("U:", u_stat, "One-sided p-value (coalition > opposition):", p_one)

# Rank-biserial effect size --> Count how many coalition scores are higher than opposition scores
favorable = sum(c > o for c, o in product(coal_scores, oppo_scores))
unfavorable = sum(c < o for c, o in product(coal_scores, oppo_scores))
r = (favorable - unfavorable) / (n1 * n2)
print("Effect size (rank-biserial r):", r)

U: 1645543.0 One-sided p-value (coalition > opposition): 1.5234517229098035e-70
Effect size (rank-biserial r): 0.3623065304147372


The one-sided test cofirms that the direction is statistically significant (coalition > opposition). The more robust interpetation of the Rank-biserial effect size also points out that the effect size (r=0.36) is medium, signifying a moderate difference in sentiment distribution between coalition and opposition.

The test is repeated, this time removing speeches made by chairspeaker

In [5]:
df_reg = df[df["Speaker_role"] == "Regular"]
df_reg.shape

(1748, 26)

### Two-sided Mann-Whitney U test, removed procedural speech

For the second run, we remove procedural speeches which can increase the number of neutral sentiment speeches and reduce the contrast between coalition and opposition sentiment.

In [6]:
#Two-sided Mann-Whitney U test, removed procedural speech

coal = df_reg[df_reg["Party_status"] == "Coalition"]["Senti_n"]
oppo = df_reg[df_reg["Party_status"] == "Opposition"]["Senti_n"]

u_stat, p_two = mannwhitneyu(coal, oppo, alternative="two-sided")
print("Mann–Whitney U statistic:", u_stat)
print("Two-sided p-value:", p_two)

favorable = sum(c > o for c, o in product(coal, oppo))
unfavorable = sum(c < o for c, o in product(coal, oppo))
r = (favorable - unfavorable) / (n1 * n2)
print("Effect size (rank-biserial r):", r)

print("Coalition median:", coal.median())
print("Opposition median", oppo.median())


Mann–Whitney U statistic: 380540.5
Two-sided p-value: 2.711033572845041e-54
Effect size (rank-biserial r): 0.10152457613753348
Coalition median: 1.435
Opposition median 0.19


- The p-value shows highly significant difference between distributions, even after excluding procedural speeches.
- The effect size r = 0.10 indicates the difference is statistically reliable but small in magnitude.
- The medians suggest coalition still has higher sentiment than opposition, consistent with your hypothesis — but the difference is less pronounced when procedural content is excluded.

### The two-sided test with removed procedural speech

In [7]:
u_stat, p_one = mannwhitneyu(coal, oppo, alternative="greater")
print("Mann–Whitney U statistic:", u_stat)
print("One-sided p-value:", p_one)

Mann–Whitney U statistic: 380540.5
One-sided p-value: 1.3555167864225205e-54


The one-sided test cofirms that the direction is statistically significant (coalition > opposition), even after excluding procedural speeches. 

## H2: Changes in political orientation will correlate with sentiment shifts, with right/traditionalist parties showing higher negativity.

The statistical checks for the H2 will consist of: 
- Kruskal–Wallis (non-parametric, can compare >2 groups) to test the distributional differences in sentiment across several groups
- Spearman correlation test on CHES data (lrgen, galtan) to test the correlation between them.
- As CHES data include many data gaps (years), we will use the Spearman correlation test to supplement the Kruskal-Wallis test together with line graphs to plot time trajectories.


As with Mann-Whitney U, we will run the Kruskal-Wallis test twice, once on full dataset 

In [8]:
# Kruskal-Wallis on full dataset

from scipy.stats import kruskal, spearmanr
df = df[df["Party_orientation"] != "-"] #Remove speeches without affiliation 
groups = [group["Senti_n"].values for name, group in df.groupby("Party_orientation")]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H = {stat:.3f}, p = ", p)

medians = df.groupby("Party_orientation")["Senti_n"].median().sort_values()
counts = df.groupby("Party_orientation")["Senti_n"].count().sort_values()

print("\nMedians by orientation:", medians)
print("\nNumber of speeches per orientation:", counts)
print("All speeches: ", len(df))

Kruskal-Wallis H = 223.808, p =  1.0256892362708425e-44

Medians by orientation: Party_orientation
Left                      0.020
Centre-right              1.590
Centre-left               2.525
Centre                    2.680
Right                     2.900
Centre to centre-left     2.960
Right to far-right        2.990
Centre to centre-right    3.000
Name: Senti_n, dtype: float64

Number of speeches per orientation: Party_orientation
Centre                      14
Left                       101
Centre to centre-right     161
Centre-right               212
Right to far-right         235
Centre-left                532
Right                      847
Centre to centre-left     1029
Name: Senti_n, dtype: int64
All speeches:  3131


In [9]:
# Kruskal-Wallis on dataset with removed procedural speech
df_reg = df_reg[df_reg["Party_orientation"] != "-"] # Remove speeches without affiliation 

groups = [group["Senti_n"].values for name, group in df_reg.groupby("Party_orientation")]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H = {stat:.3f}, p = ", p)

medians = df_reg.groupby("Party_orientation")["Senti_n"].median().sort_values()
counts = df_reg.groupby("Party_orientation")["Senti_n"].count().sort_values()

print("\nMedians by orientation:", medians)
print("\nNumber of speeches per orientation:", counts)
print("All speeches: ", len(df_reg))

Kruskal-Wallis H = 140.913, p =  3.2724118630017385e-27

Medians by orientation: Party_orientation
Left                      0.020
Right to far-right        0.210
Right                     0.390
Centre to centre-left     0.510
Centre-left               0.580
Centre-right              1.085
Centre to centre-right    1.185
Centre                    2.680
Name: Senti_n, dtype: float64

Number of speeches per orientation: Party_orientation
Centre                     14
Centre to centre-right     42
Right to far-right         93
Left                      101
Centre-right              192
Centre-left               273
Centre to centre-left     375
Right                     375
Name: Senti_n, dtype: int64
All speeches:  1465


Observations (sample):
The Kruskal-Wallis test shows that there are statistically significant differences between the speeches of parties of different political orientations both on full dataset (H = 223.808, p =  1.02x 10<sup>-44</sup> ) as well as on dataset with removed procedural speeches (H = 140.913, p =  3.27x10<sup>-27</sup>). For the original dataset, the median and counts reflect the following: 
- Trend: Very clear monotonic increase:
- Left = strongly negative.
- Centre-left/centre = moderately positive.
- Right and far-right = most positive.
Sample sizes:
- Large groups (e.g., Right = 847, Centre-to-centre-left = 1029) → stable medians.
- Only Centre = 14 is tiny, but it fits the general upward trend anyway.
Interpretation:
With all speeches, you get strong support for H2 as you originally phrased it — Right/traditionalist parties are more positive, Left more negative.


The median sentiment score was lowest for Left (0.02) and Right to far-right (0.21) parties, while centrist parties expressed a more positive or neutral sentiment (Centre = 2.68). This suggests a U-shaped pattern: parties at the ideological extremes (left and far right) tend to be more negative, while centrist parties tend to be more positive.
However, the Centre category only contained very few speeches (N = 14), so this median is less reliable. The results for larger groups (N > 100) support the general finding that both extremes are more negative than the centre.”

- U-shaped medians: Left = negative, Centre = positive, Right = back to negative.
- Right and Far-right medians collapsed from ~3.0 (full dataset) → ~0.2–0.4 (after removing).
- This reversed the story, suggesting centrists are more positive, extremes more negative. This indicates that neutral/procedural speeches, which tend to be disproportionately associated with right-wing orientations, strongly influence overall sentiment trends.”

## Spearman correlation of the lrgen and galtan variables
A supporting statistical check, Spearman correlation checks whether the general left-right positioning is also associated with the cultural positioning of parties. 

In [12]:
ches = pd.read_csv("../Results/Datasets/CHES_Family.tsv", sep="\t", encoding="utf-8")
ches.head()

,parlamint,party_id,party,year,lrgen,galtan,family,seat,grouped_parties
0,DeSUS,2906,DeSUS,2002,3.4,5.8,No family,4.4,DeSUS
1,DeSUS,2906,DeSUS,2006,3.2,4.5,No family,4.4,DeSUS
2,DeSUS,2906,DeSUS,2010,4.2,5.2,No family,7.8,DeSUS
3,DeSUS,2906,DeSUS,2014,4.2,5.3,No family,11.1,DeSUS
4,DeSUS,2906,DeSUS,2019,3.8,5.1,No family,5.7,DeSUS


In [13]:
rho, pval = spearmanr(ches["lrgen"], ches["galtan"])
print(f"Spearman correlation lrgen–galtan: rho = {rho:.3f}, p = {pval:.3e}")


Spearman correlation lrgen–galtan: rho = 0.824, p = 6.709e-11


The correlation check shows very strong positive correlation between lrgen and galtan and that parties positioned further to the right (lrgen) also tend to be more traditional/authoritarian (galtan).